In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, StandardScaler, RobustScaler, KBinsDiscretizer





TASK 1

In [ ]:
df = pd.read_excel("data/Online_Retail.xlsx")

print(df.shape)

# checking too see what has missing values  
df.isna().sum()

Task 1 (Numerical): Missing Value Imputation (sklearn.impute.SimpleImputer): For numerical streams (Quantity, UnitPrice), impute using the median strategy to prevent outlier distortion.


In [ ]:
#making list of the columns we wish to transform
cols_to_transform = ["Quantity", "UnitPrice"]

#create imputer
imputer = SimpleImputer(strategy="median")

#apply transformation to the correnct columns
df[cols_to_transform] = imputer.fit_transform(df[cols_to_transform])

print(df[cols_to_transform].isnull().sum())

Task 2: Continuous Feature Normalization (sklearn.preprocessing): Implement StandardScaler (Z-score centering to zero mean and unit variance). Contrast performance against MinMaxScaler [0, 1] and RobustScaler (IQR), documenting sensitivity to extreme transaction anomalies

In [ ]:
#first we import the normalization functions for the sake of not cluttering my code
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

In [ ]:
#we now have to transform the data using normalization formulats
# Making copies for later comparison
df_standard = df.copy()
df_minmax = df.copy()
df_robust = df.copy()

# we then transform them using the normalization preprocessing in the order mentioned in the assignment
df_standard[cols_to_transform] = StandardScaler().fit_transform(df[cols_to_transform])
df_minmax[cols_to_transform] = MinMaxScaler().fit_transform(df[cols_to_transform])
df_robust[cols_to_transform] = RobustScaler().fit_transform(df[cols_to_transform])

Once the transformations are done we have to contrast the performance of the sensitivty to extreme anomalies


In [ ]:
print("Untransformed Dataframe")
print(df[cols_to_transform].describe())

In [ ]:
print("Standard Scaler transformed data")
print(df_standard[cols_to_transform].describe())

In [ ]:
print("MinMax Scaler transformed data")
print(df_minmax[cols_to_transform].describe())

In [ ]:
print("Robust Scaler transformed data")
print(df_robust[cols_to_transform].describe())

As we know ZScore centers the mean to zero and standard deviation to 1. Issues with this is that Really big outliers still control the mean. As you can see in the Standard scale description for both quantity and unit price. The mean is an extremely small number for both of them means for quantity: 3.041948e-18, UnitPrice: -1.006990e-17 but the std is 1 for both of them. This means that this normalization method has high sensitivity to extreme outliers.

For the minMax normalization we run into a different problem and that is that the data stays more or less the same. since now the max is one and the min 0 extreme outliers would push most of the data to a smaller range of the space. We can see this in the MinMax scaler describe funstion. Significant portions of the data are only within the first 25% distribution line. This means that this nomalization method is has high sensitivity to outiliers.

For the Robust Scaler it tries to put most of the data between the first 3 quarters in this case 25%-75%. We can see this succesfully in quantity and not so succesfully in UnitPrice. With that being said, when compared to the original retail data the distribution has remained fairly similar. Almost scaled down by a factor of 10. Since the distribution of the data remains fairly unchanged I think that the Robust scaler has low sensitivity to outliers as the data stays similarly distributed.

Task 3: Non-Linear Feature Discretization (sklearn.preprocessing.KBinsDiscretizer): Map continuous pricing data into discrete ordinal bins. Compare (1) Uniform (equal-width), (2) Quantile (equal-frequency), and (3) K-means binning strategies.


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
#so for this part we are using only the continuous feature which is unit price so we make a copy of that

df_copy = df.copy()
prices = df_copy[["UnitPrice"]]


In [ ]:
#reading kbinDiscretizer We have to encode with ordinal to keep it as integer

uniform = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="uniform")
quantile = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="quantile")
kmeans = KBinsDiscretizer(n_bins=12, encode="ordinal", strategy="kmeans")

In [ ]:
df_uniform = df.copy()
df_quantile = df.copy()
df_kmeans = df.copy()

# we then transform them using the normalization preprocessing in the order mentioned in the assignment
df_uniform[["UnitPrice"]] = uniform.fit_transform(df[["UnitPrice"]])
df_quantile[["UnitPrice"]] = quantile.fit_transform(df[["UnitPrice"]])
df_kmeans[["UnitPrice"]] = kmeans.fit_transform(df[["UnitPrice"]])

In [ ]:
print("Uniform")
print(df_uniform[["UnitPrice"]].value_counts().sort_index())

In [ ]:
print("\nQuantile")
print(df_quantile[["UnitPrice"]].value_counts().sort_index())

In [ ]:
print("\nKMeans")
print(df_kmeans[["UnitPrice"]].value_counts().sort_index())

Comparing the 3 binning strategies yields the previous description.

The uniform method seems to have places most datapoints in bin 2 and then the rest of the bins received minimal elements. It did not even use all of the bins. It did not distribute the data well as it all stayed in one bin

The quantile method seems to have distributed the bins fairly equally. This method seems to balance how many items there are in each bin effectively balancing the entire data set.

The KMeans methods seems to have spread the population unevenly as well but every bin is being used. What this means to me is that that since kmeans explore the different means for each bin then groups items based on how much closer that mean is than the other means. Each bin is grouped with objects that have similar UnitPrice Values

unified column pipeline

In [ ]:
import numpy as np

In [ ]:
df = pd.read_excel("data/Online_Retail.xlsx")

In [ ]:

# Text Preprocessing 
df['Description'] = df['Description'].str.lower()
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.lower()
df['CustomerID'] = df['CustomerID'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# Split between Test and Train
X_train, X_test, = train_test_split(
    df, test_size=0.2, random_state=21, 
)

# Pipeline & Column Transformers 
categorical_features = ["InvoiceNo", "StockCode", "Description", "InvoiceDate","CustomerID","Country"]

# X_train[categorical_features] = X_train[categorical_features].astype(str).str.lower()
# X_test[categorical_features] = X_test[categorical_features].astype(str).str.lower()

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='constant',fill_value= 'Unknown')),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

#Numerical part of pipeline

num_pipe = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, categorical_features),
        ("numerical", num_pipe, cols_to_transform),
    ]
)

preprocessor.fit(X_train).transform(X_test)

In [ ]:


# cleaning Description 
df['Description'] = (
    df['Description']
      .str.lower()
      .str.replace(r'[^\w\s]', ' ', regex=True)
      .str.replace(r'[\s+]', ' ', regex=True)
      .str.strip()
      .replace('', np.nan)
)

# Derive isCancellation from invoice 
df['IsCancellation'] = df['InvoiceNo'].astype(str).str.upper().str.startswith('C')

# Derive Month and DayOfWeek 

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Month'] = df['InvoiceDate'].dt.month          
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek  

# Derive HasCustomerID
df['HasCustomerID'] = df['CustomerID'].notna()

# converet derived strings too int
df['IsCancellation'] = df['IsCancellation'].astype(int)
df['HasCustomerID'] = df['HasCustomerID'].astype(int)


# X_train[categorical_features] = X_train[categorical_features].astype(str)
# X_test[categorical_features] = X_test[categorical_features].astype(str)

# Split between Test and Train
X_train, X_test, = train_test_split(
    df, test_size=0.2, random_state=21, 
)

# Pipeline & Column Transformers 
categorical_features = ['Description', 'Country', 'Month', 'DayOfWeek', 'IsCancellation', 'HasCustomerID']

categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy='constant',fill_value= 'Unknown')),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipe, categorical_features),
    ],remainder='drop'
)

preprocessor.fit(X_train).transform(X_test)